In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")
    
MODEL = "openai/gpt-oss-120b"
groq = OpenAI(base_url="https://api.groq.com/openai/v1",api_key=groq_api_key)

In [ ]:
system_message = """
You are a personal AI assistant with the energy of an over-caffeinated best friend who somehow gets things done.
Keep responses extremely concise (1-2 lines maximum).
Always be accurate. If you don't know the answer, say so.
You have access to tools to manage the user's Google Calendar.
IMPORTANT: When the user asks about scheduling, listing, or deleting events, ALWAYS call `get_current_time` first to know what the current date and time is. Relative dates like "tomorrow", "next Monday", or "today" must be calculated based on the returned current time.
"""

### Calendar Tools
The following functions interact with Google Calendar.

In [ ]:
import datetime
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/calendar']

In [ ]:
def get_calendar_service():
    """Authenticate and return Google Calendar service."""
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists('credentials.json'):
                raise FileNotFoundError("credentials.json not found in the current directory.")
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    service = build('calendar', 'v3', credentials=creds)
    return service

In [ ]:
def get_current_time():
    """Get the current date and time in ISO format."""
    return datetime.datetime.now().astimezone().isoformat()

In [ ]:
def add_event(title, date, start_time, end_time, description=""):
    """
    Add a new event to Google Calendar.
    
    Parameters:
    title (str): Title of the event.
    date (str): Date in YYYY-MM-DD format.
    start_time (str): Start time in HH:MM (24-hour format).
    end_time (str): End time in HH:MM (24-hour format).
    description (str): Optional description.
    """
    try:
        service = get_calendar_service()
        start_dt = datetime.datetime.strptime(f"{date} {start_time}", "%Y-%m-%d %H:%M").astimezone()
        end_dt = datetime.datetime.strptime(f"{date} {end_time}", "%Y-%m-%d %H:%M").astimezone()
        
        event = {
            'summary': title,
            'description': description,
            'start': {
                'dateTime': start_dt.isoformat(),
            },
            'end': {
                'dateTime': end_dt.isoformat(),
            }
        }
        
        created_event = service.events().insert(calendarId='primary', body=event).execute()
        return f"Successfully scheduled event: '{title}' on {date} at {start_time}-{end_time}. Event ID: {created_event.get('id')}"
    except Exception as e:
        return f"Error adding event: {str(e)}"

In [ ]:
def list_events(date=None):
    """
    List events from Google Calendar.
    
    Parameters:
    date (str): Optional. Filter events for a specific date in YYYY-MM-DD format.
    """
    try:
        service = get_calendar_service()
        
        if date:
            start_dt = datetime.datetime.strptime(f"{date} 00:00:00", "%Y-%m-%d %H:%M:%S").astimezone()
            end_dt = datetime.datetime.strptime(f"{date} 23:59:59", "%Y-%m-%d %H:%M:%S").astimezone()
            time_min = start_dt.isoformat()
            time_max = end_dt.isoformat()
        else:
            now = datetime.datetime.now().astimezone()
            time_min = now.isoformat()
            time_max = (now + datetime.timedelta(days=7)).isoformat()
            
        events_result = service.events().list(
            calendarId='primary', 
            timeMin=time_min,
            timeMax=time_max,
            singleEvents=True,
            orderBy='startTime'
        ).execute()
        
        events = events_result.get('items', [])
        
        if not events:
            return f"No events found for {date}." if date else "No events found for the next 7 days."
            
        output = []
        for e in events:
            start = e['start'].get('dateTime', e['start'].get('date'))
            end = e['end'].get('dateTime', e['end'].get('date'))
            desc = f" ({e.get('description')})" if e.get('description') else ""
            output.append(f"ID: {e['id']} | Start: {start} | End: {end} | {e['summary']}{desc}")
            
        return "\n".join(output)
    except Exception as e:
        return f"Error listing events: {str(e)}"

In [ ]:
def delete_event(event_id):
    """
    Delete an event from Google Calendar.
    
    Parameters:
    event_id (str): The unique Google Calendar event ID.
    """
    try:
        service = get_calendar_service()
        service.events().delete(calendarId='primary', eventId=event_id).execute()
        return f"Successfully deleted event (ID: {event_id})"
    except Exception as e:
        return f"Error deleting event: {str(e)}"

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current date and time in ISO format to anchor relative dates like 'today', 'tomorrow', 'next week' etc.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_event",
            "description": "Add a new event to Google Calendar.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "The title of the event."},
                    "date": {"type": "string", "description": "The date of the event in YYYY-MM-DD format."},
                    "start_time": {"type": "string", "description": "The start time of the event in HH:MM format (24-hour format)."},
                    "end_time": {"type": "string", "description": "The end time of the event in HH:MM format (24-hour format)."},
                    "description": {"type": "string", "description": "Optional description of the event."}
                },
                "required": ["title", "date", "start_time", "end_time"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_events",
            "description": "List events scheduled on Google Calendar. Filters by a specific date, or lists upcoming 7 days if no date filter is provided.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {"type": "string", "description": "Optional date filter in YYYY-MM-DD format."}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "delete_event",
            "description": "Delete a calendar event using its unique Google Calendar event ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "event_id": {"type": "string", "description": "The unique event ID."}
                },
                "required": ["event_id"]
            }
        }
    }
]

In [ ]:
def chat(message, history):
    history_messages = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history_messages + [{"role": "user", "content": message}]
    
    max_iterations = 5
    iteration = 0
    
    while iteration < max_iterations:
        response = groq.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
        
        response_message = response.choices[0].message
        
        if response_message.tool_calls:
            messages.append(response_message)
            
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"Executing Tool: {function_name} with args {function_args}")
                
                if function_name == "get_current_time":
                    result = get_current_time()
                elif function_name == "add_event":
                    result = add_event(**function_args)
                elif function_name == "list_events":
                    result = list_events(**function_args)
                elif function_name == "delete_event":
                    result = delete_event(**function_args)
                else:
                    result = f"Error: Tool '{function_name}' not found."
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result
                })
            iteration += 1
        else:
            return response_message.content
            
    return "Error: Maximum tool call iterations reached."

gr.ChatInterface(fn=chat, type="messages").launch()